# Phishing Website & URL Detector Using Machine Learning

**IBM SkillsBuild Cyber Security 2026 — Project Notebook**  
**Student:** YourName

## Project overview
This notebook trains a machine-learning classifier to distinguish phishing URLs from legitimate URLs using URL-text features only. It uses the public PhiUSIIL dataset from the UCI Machine Learning Repository, then extracts lexical features from the URL strings.

**Safety note:** This is an educational risk-screening tool, not a guarantee that any URL is safe. Do not open suspicious URLs to test them. The notebook does not visit or contact the URLs in the dataset.

## What you will do
1. Install/import the required libraries.
2. Download the dataset through the UCI repository API.
3. inspect and clean the URL/label data.
4. Extract features from URL text.
5. Split the data into training and test sets.
6. Train and evaluate a Random Forest classifier.
7. Try the trained model on a URL string you enter.

## 1. Install dependencies

If you are running this in a fresh Jupyter environment, run the following command in a notebook cell. In Google Colab, the required packages are often already installed.

In [ ]:
%pip install -q ucimlrepo pandas numpy scikit-learn matplotlib seaborn

## 2. Import libraries and set reproducibility options

In [ ]:
import re
import ipaddress
from urllib.parse import urlparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 3. Download the public dataset

Dataset: **PhiUSIIL Phishing URL (Website)**  
Official page: https://archive.ics.uci.edu/dataset/967/phiusiil-phishing-url-dataset

The original dataset has 235,795 records. This notebook uses the URL text and the official target label. Other dataset features may include information that cannot be derived from URL text alone, so they are not used by this model.

In [ ]:
dataset = fetch_ucirepo(id=967)

X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.copy()

print("Feature columns:", list(X_raw.columns)[:15])
print("Target columns:", list(y_raw.columns))
print("Feature table shape:", X_raw.shape)
print("Target table shape:", y_raw.shape)

# Find the URL column without assuming capitalization.
url_candidates = [c for c in X_raw.columns if str(c).strip().lower() == "url"]
if not url_candidates:
    raise ValueError(
        "Could not find a URL column in the downloaded dataset. "
        "Check the UCI dataset page and update the URL column selection."
    )

url_col = url_candidates[0]
df = pd.DataFrame({
    "url": X_raw[url_col].astype(str),
    "label": y_raw.iloc[:, 0]
})

# Dataset documentation: label 1 = legitimate; label 0 = phishing.
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["url", "label"])
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)
df = df[df["url"].str.strip().ne("")].drop_duplicates(subset=["url"]).reset_index(drop=True)

print(f"Usable unique URL records: {len(df):,}")
print("Label counts (0 = phishing, 1 = legitimate):")
print(df["label"].value_counts().sort_index())
df.head()

## 4. Inspect the class balance

A class-balance chart helps us see how many phishing and legitimate examples are in the usable dataset.

In [ ]:
label_names = {0: "Phishing", 1: "Legitimate"}
class_counts = df["label"].map(label_names).value_counts()

ax = class_counts.plot(kind="bar", rot=0, figsize=(6, 4))
ax.set_title("Dataset class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of URLs")
plt.tight_layout()
plt.show()

## 5. Extract features from URL text

These features are calculated from the URL string only. The code does **not** connect to the URL or load the webpage. Features include URL length, hostname length, number of dots, digits, hyphens and special characters, use of an IP address as hostname, HTTPS scheme, and suspicious-looking tokens.

These are simple indicators, not definitive proof of maliciousness.

In [ ]:
SUSPICIOUS_TOKENS = [
    "login", "verify", "update", "secure", "account", "banking",
    "confirm", "password", "signin", "wallet", "support", "auth"
]

def _has_port(parsed_url):
    """Safely detect an explicitly parsed port, including malformed URL edge cases."""
    try:
        return parsed_url.port is not None
    except ValueError:
        return False

def extract_url_features(url):
    """Return numeric features derived from URL text without visiting it."""
    raw = str(url).strip()
    candidate = raw if "://" in raw else "http://" + raw

    try:
        parsed = urlparse(candidate)
        hostname = parsed.hostname or ""
        path = parsed.path or ""
    except Exception:
        hostname, path, parsed = "", "", urlparse("")

    lower = raw.lower()
    try:
        ipaddress.ip_address(hostname)
        host_is_ip = 1
    except ValueError:
        host_is_ip = 0

    return {
        "url_length": len(raw),
        "hostname_length": len(hostname),
        "path_length": len(path),
        "count_dots": raw.count("."),
        "count_hyphens": raw.count("-"),
        "count_at": raw.count("@"),
        "count_question": raw.count("?"),
        "count_equal": raw.count("="),
        "count_ampersand": raw.count("&"),
        "count_digits": sum(ch.isdigit() for ch in raw),
        "count_letters": sum(ch.isalpha() for ch in raw),
        "count_percent": raw.count("%"),
        "count_slashes": raw.count("/"),
        "count_underscore": raw.count("_"),
        "count_non_alnum": sum(not ch.isalnum() for ch in raw),
        "https_scheme": int(parsed.scheme.lower() == "https"),
        "host_is_ip": host_is_ip,
        "has_port": int(_has_port(parsed)),
        "subdomain_count": max(hostname.count(".") - 1, 0),
        "suspicious_token_count": sum(token in lower for token in SUSPICIOUS_TOKENS),
        "has_punycode": int("xn--" in lower),
        "has_double_slash_in_path": int("//" in path),
    }

features_df = pd.DataFrame(df["url"].apply(extract_url_features).tolist())
labels = df["label"].copy()

print("Extracted feature matrix shape:", features_df.shape)
features_df.head()

## 6. Create training and test sets

A stratified split preserves approximately the same class proportions in both sets. The test set is kept separate from model fitting and is used for the final evaluation.

To keep this beginner project manageable on ordinary computers, the notebook uses a reproducible sample of up to 60,000 URLs. Change `MAX_ROWS` if your computer has enough memory and processing time.

In [ ]:
MAX_ROWS = 60_000

model_data = pd.concat(
    [features_df.reset_index(drop=True), labels.reset_index(drop=True).rename("label")],
    axis=1
)

if len(model_data) > MAX_ROWS:
    model_data = model_data.groupby("label", group_keys=False).apply(
        lambda group: group.sample(
            n=max(1, round(MAX_ROWS * len(group) / len(model_data))),
            random_state=RANDOM_STATE
        )
    ).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X = model_data.drop(columns=["label"])
y = model_data["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training class proportions:")
print(y_train.value_counts(normalize=True).sort_index().rename(index=label_names))

## 7. Train the Random Forest model

Random Forest combines many decision trees. It is a useful baseline for tabular numeric features and does not require feature scaling.

In [ ]:
model = RandomForestClassifier(
    n_estimators=150,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1,
    min_samples_leaf=2
)

model.fit(X_train, y_train)
print("Model training complete.")

## 8. Evaluate the model on the held-out test set

The metrics below are calculated by running this notebook. No accuracy or performance result is assumed in advance.

- **Precision for phishing (label 0):** among URLs flagged as phishing, how many were phishing in this test set?
- **Recall for phishing (label 0):** among phishing URLs in the test set, how many did the model identify?
- **F1-score:** combines precision and recall.
- **Accuracy:** proportion of all test examples classified correctly.

High test performance on this dataset does not guarantee the same performance on new or future phishing campaigns.

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (phishing class 0): {precision_score(y_test, y_pred, pos_label=0, zero_division=0):.4f}")
print(f"Recall (phishing class 0):    {recall_score(y_test, y_pred, pos_label=0, zero_division=0):.4f}")
print(f"F1-score (phishing class 0):  {f1_score(y_test, y_pred, pos_label=0, zero_division=0):.4f}")
print()
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Phishing", "Legitimate"],
    zero_division=0
))

## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Phishing", "Legitimate"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix — held-out test set")
plt.tight_layout()
plt.show()

## 10. Most influential features

Feature importance is a model-based summary of which input features were useful to the trained forest. It does not prove that any feature causes phishing.

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(12).to_frame("importance"))

importance.head(12).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 12 Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Test a URL string

Enter a URL you want to inspect. The notebook only analyzes its text; it does not visit the site. Do not paste passwords, tokens, private links, or personal information.

The output is a model prediction, not a safety certification. A URL predicted as legitimate can still be dangerous.

In [ ]:
def predict_url_risk(url, trained_model=model):
    row = pd.DataFrame([extract_url_features(url)])
    prediction = int(trained_model.predict(row)[0])
    probabilities = trained_model.predict_proba(row)[0]
    class_to_index = {int(c): i for i, c in enumerate(trained_model.classes_)}
    phishing_probability = float(probabilities[class_to_index[0]])

    if prediction == 0:
        verdict = "POTENTIAL PHISHING — treat with caution"
    else:
        verdict = "Predicted legitimate by this model — still verify independently"

    return {
        "input_url": url,
        "prediction": verdict,
        "estimated_phishing_probability": round(phishing_probability, 4),
        "note": "Text-based model output only; not a guarantee of safety."
    }

# Replace this example with a URL string you have permission to assess.
example_url = "https://example.com/account"
predict_url_risk(example_url)

## 12. Save the trained model (optional)

This cell saves the model and the ordered feature names for later local use. Only load pickle/joblib files that you created or trust; these formats can execute code when loaded.

In [ ]:
import joblib

joblib.dump(
    {"model": model, "feature_names": list(X.columns)},
    "phishing_url_detector.joblib"
)
print("Saved: phishing_url_detector.joblib")

## 13. Conclusion

This project demonstrates a complete supervised machine-learning workflow: public dataset retrieval, URL-text feature engineering, stratified train/test split, model training, evaluation, and prediction for a new URL string.

### Limitations
- URL lexical patterns change as attackers adapt.
- The training data may not represent all languages, regions, URL shorteners, or newly created attacks.
- A text-only model does not inspect webpage content, redirects, certificate details, or reputation feeds.
- Test results from a random split may be optimistic if related URLs occur in both partitions.
- A “legitimate” prediction must never be treated as proof that a site is safe.

### Dataset citation
Prasad, A. & Chandra, S. (2024). *PhiUSIIL: Phishing URL (Website) Dataset*. UCI Machine Learning Repository. https://doi.org/10.1016/j.cose.2023.103545  
Dataset page: https://archive.ics.uci.edu/dataset/967/phiusiil-phishing-url-dataset